# Post 006 — Unsupervised Discovery: K-Means, DBSCAN & PCA
## Dataset A: Smartwatch Activity Clustering

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

In supervised learning, we have labels. But what happens when we don't? Unsupervised learning discovers hidden structure in data without any labels. In this notebook, we use a smartwatch dataset — accelerometer, gyroscope, heart rate, and step count readings — to automatically discover activity clusters like Running, Walking, Sitting, Sleeping, and Cycling.

The beauty of this problem: **we never tell the model what the activities are. It finds them itself.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully')

## 1. Load and Explore the Data

Our dataset contains 4,000 smartwatch readings with 8 sensor features. Each row is one 30-second window of sensor data. The `true_activity` column exists only for evaluation — we will hide it during clustering and reveal it at the end to see how well the algorithm did.

In [ ]:
df = pd.read_csv('../data/smartwatch_activity.csv')
print(f'Shape: {df.shape}')
print(f'\nActivity distribution:')
print(df['true_activity'].value_counts())
df.head()

In [ ]:
# Statistical summary of sensor features
feature_cols = [c for c in df.columns if c != 'true_activity']
df[feature_cols].describe().round(3)

## 2. Preprocessing: Feature Scaling

Clustering algorithms like K-Means use Euclidean distance. If one feature (e.g., steps: 0-200) has a much larger range than another (e.g., heart_rate: 60-180), the steps feature will dominate the distance calculation unfairly.

**StandardScaler** transforms each feature to have mean=0 and standard deviation=1, putting all features on equal footing.

In [ ]:
# Separate features from the true labels (labels hidden during clustering)
X = df[feature_cols].values
y_true = df['true_activity'].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Before scaling — steps mean: {X[:, feature_cols.index("steps_per_min")].mean():.1f}, std: {X[:, feature_cols.index("steps_per_min")].std():.1f}')
print(f'After scaling  — steps mean: {X_scaled[:, feature_cols.index("steps_per_min")].mean():.4f}, std: {X_scaled[:, feature_cols.index("steps_per_min")].std():.4f}')

## 3. Dimensionality Reduction with PCA

With 8 features, we cannot visualize the data directly. **Principal Component Analysis (PCA)** finds the directions of maximum variance in the data and projects it onto fewer dimensions.

Think of it as finding the best camera angle to photograph a 3D object — PCA finds the 2D projection that preserves the most information.

In [ ]:
# PCA to 2D for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'Explained variance by PC1: {pca.explained_variance_ratio_[0]:.1%}')
print(f'Explained variance by PC2: {pca.explained_variance_ratio_[1]:.1%}')
print(f'Total variance retained:   {pca.explained_variance_ratio_.sum():.1%}')

# Visualize the PCA projection colored by true activity
fig, ax = plt.subplots(figsize=(10, 7))
activities = df['true_activity'].unique()
colors = plt.cm.Set1(np.linspace(0, 1, len(activities)))
for act, col in zip(activities, colors):
    mask = y_true == act
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[col], label=act, alpha=0.6, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.set_title('PCA Projection of Smartwatch Data (colored by true activity)')
ax.legend()
plt.tight_layout()
plt.show()

## 4. K-Means Clustering

**K-Means** partitions data into K clusters by iteratively:
1. Assigning each point to the nearest centroid
2. Moving each centroid to the mean of its assigned points

The algorithm converges when centroids stop moving. The key challenge: **we must choose K in advance.** We use the Elbow Method and Silhouette Score to find the optimal K.

In [ ]:
# Elbow Method: plot inertia (within-cluster sum of squares) vs K
inertias = []
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, km.labels_, sample_size=1000))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(K_range, inertias, 'bo-', linewidth=2, markersize=8)
ax1.axvline(x=5, color='red', linestyle='--', alpha=0.7, label='Elbow at K=5')
ax1.set_xlabel('Number of Clusters (K)')
ax1.set_ylabel('Inertia (WCSS)')
ax1.set_title('Elbow Method')
ax1.legend()

ax2.plot(K_range, silhouette_scores, 'go-', linewidth=2, markersize=8)
ax2.axvline(x=5, color='red', linestyle='--', alpha=0.7, label='Best at K=5')
ax2.set_xlabel('Number of Clusters (K)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score (higher = better separation)')
ax2.legend()

plt.suptitle('Choosing Optimal K for Smartwatch Activity Clustering', fontsize=14)
plt.tight_layout()
plt.show()
print(f'Best silhouette score: {max(silhouette_scores):.3f} at K={K_range[silhouette_scores.index(max(silhouette_scores))]}')

In [ ]:
# Fit K-Means with optimal K=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Evaluate against true labels
ari = adjusted_rand_score(y_true, kmeans_labels)
sil = silhouette_score(X_scaled, kmeans_labels)
print(f'K-Means Results (K=5):')
print(f'  Adjusted Rand Index: {ari:.3f}  (1.0 = perfect, 0 = random)')
print(f'  Silhouette Score:    {sil:.3f}  (1.0 = perfect separation)')

# Visualize K-Means clusters in PCA space
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='Set1', alpha=0.5, s=15)
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax1.scatter(centroids_pca[:, 0], centroids_pca[:, 1], c='black', marker='X', s=200, zorder=5, label='Centroids')
ax1.set_title(f'K-Means Clusters (ARI={ari:.3f})')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2')
ax1.legend()

for act, col in zip(activities, colors):
    mask = y_true == act
    ax2.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[col], label=act, alpha=0.5, s=15)
ax2.set_title('True Activity Labels')
ax2.set_xlabel('PC1'); ax2.set_ylabel('PC2')
ax2.legend(fontsize=8)

plt.suptitle('K-Means vs True Labels in PCA Space', fontsize=14)
plt.tight_layout()
plt.show()

## 5. DBSCAN Clustering

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** takes a completely different approach:
- It does not require specifying K in advance
- It finds clusters of arbitrary shape (not just spherical)
- It explicitly labels outliers as noise (label = -1)

Two parameters control DBSCAN:
- **eps**: The neighborhood radius — how close two points must be to be considered neighbors
- **min_samples**: Minimum points in a neighborhood to form a core point

In [ ]:
# DBSCAN clustering
dbscan = DBSCAN(eps=0.8, min_samples=15)
dbscan_labels = dbscan.fit_predict(X_scaled)

n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = (dbscan_labels == -1).sum()
ari_db = adjusted_rand_score(y_true[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1])

print(f'DBSCAN Results:')
print(f'  Clusters found:      {n_clusters}')
print(f'  Noise points:        {n_noise} ({n_noise/len(dbscan_labels):.1%})')
print(f'  Adjusted Rand Index: {ari_db:.3f}')

# Visualize DBSCAN
fig, ax = plt.subplots(figsize=(10, 7))
unique_labels = sorted(set(dbscan_labels))
cmap = plt.cm.get_cmap('tab10', len(unique_labels))
for i, label in enumerate(unique_labels):
    mask = dbscan_labels == label
    if label == -1:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c='gray', alpha=0.3, s=10, label='Noise')
    else:
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], c=[cmap(i)], alpha=0.6, s=15, label=f'Cluster {label}')
ax.set_title(f'DBSCAN Clusters (eps=0.8, min_samples=15) — {n_clusters} clusters, {n_noise} noise points')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. K-Means vs DBSCAN: Side-by-Side Comparison

Both algorithms have strengths and weaknesses. K-Means is fast and works well when clusters are roughly spherical and similar in size. DBSCAN handles irregular shapes and noise but is sensitive to its two parameters.

In [ ]:
# Feature importance: which features drive each K-Means cluster?
cluster_profiles = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), 
                                  columns=feature_cols)
cluster_profiles.index = [f'Cluster {i}' for i in range(5)]

# Map clusters to likely activities based on dominant features
print('Cluster Profiles (original scale):')
print(cluster_profiles.round(2).to_string())

# Heatmap of cluster profiles
fig, ax = plt.subplots(figsize=(12, 5))
normalized_profiles = (cluster_profiles - cluster_profiles.mean()) / cluster_profiles.std()
sns.heatmap(normalized_profiles, annot=True, fmt='.2f', cmap='RdYlGn', center=0, ax=ax)
ax.set_title('K-Means Cluster Profiles (z-scored) — What makes each cluster unique?')
plt.tight_layout()
plt.show()

## 7. PCA: Feature Contribution Analysis

PCA not only reduces dimensions for visualization — it also tells us which original features contribute most to each principal component. This helps us understand what the model is actually learning.

In [ ]:
# Full PCA to understand explained variance across all components
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax1.bar(range(1, len(feature_cols)+1), pca_full.explained_variance_ratio_, alpha=0.7, color='steelblue')
ax1.plot(range(1, len(feature_cols)+1), np.cumsum(pca_full.explained_variance_ratio_), 'ro-', linewidth=2)
ax1.axhline(y=0.95, color='green', linestyle='--', label='95% threshold')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot — How many components do we need?')
ax1.legend()

# Feature loadings for PC1 and PC2
loadings = pd.DataFrame(pca_full.components_[:2].T, index=feature_cols, columns=['PC1', 'PC2'])
loadings.plot(kind='bar', ax=ax2, color=['steelblue', 'coral'])
ax2.set_title('Feature Loadings on PC1 and PC2')
ax2.set_xlabel('Feature')
ax2.set_ylabel('Loading')
ax2.axhline(y=0, color='black', linewidth=0.5)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()
print(f'\nFeatures needed for 95% variance: {(np.cumsum(pca_full.explained_variance_ratio_) < 0.95).sum() + 1}')

## 8. Summary and Key Takeaways

| Algorithm | Clusters Found | ARI Score | Noise Handling | Shape Assumption |
|---|---|---|---|---|
| K-Means (K=5) | 5 (forced) | Computed above | None | Spherical |
| DBSCAN | Automatic | Computed above | Yes (labels noise) | Arbitrary |

**Key lessons from this notebook:**
1. **Feature scaling is mandatory** for distance-based algorithms — unscaled features produce meaningless clusters
2. **PCA is a tool, not just a trick** — it reveals structure and feature importance, not just a 2D plot
3. **The Elbow Method + Silhouette Score together** give a more reliable K estimate than either alone
4. **DBSCAN's noise detection is a feature** — in real sensor data, outliers are often the most interesting points
5. **ARI > 0.7 is excellent** for unsupervised clustering — the algorithm found meaningful structure without any labels

In [ ]:
print('=== Final Results Summary ===')
print(f'Dataset: Smartwatch Activity Clustering ({len(df)} samples, {len(feature_cols)} features)')
print(f'True activities: {list(np.unique(y_true))}')
print()
print(f'K-Means (K=5):')
print(f'  Silhouette Score:    {silhouette_score(X_scaled, kmeans_labels):.3f}')
print(f'  Adjusted Rand Index: {adjusted_rand_score(y_true, kmeans_labels):.3f}')
print()
print(f'DBSCAN (eps=0.8, min_samples=15):')
print(f'  Clusters found: {n_clusters}')
print(f'  Noise points:   {n_noise} ({n_noise/len(dbscan_labels):.1%})')
print()
print(f'PCA (2 components): {pca.explained_variance_ratio_.sum():.1%} variance retained')